In [49]:
import math
import re
import regex

from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn

### Unicode Standard vs Unicode Encodings

In [11]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print('*1* ', utf8_encoded)
print('*2* ', type(utf8_encoded))
# Get the byte values for the encoded string (integers from 0 to 255).
print('*3* ', list(utf8_encoded))
# One byte does not necessarily correspond to one Unicode character!
print('*4* ', len(test_string))
print('*5* ', len(utf8_encoded))
print('*6* ', utf8_encoded.decode("utf-8"))

*1*  b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
*2*  <class 'bytes'>
*3*  [104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
*4*  13
*5*  23
*6*  hello! こんにちは!


In [ ]:
# Unicode Standard
print(ord('ب'))
print(chr(1576))

1576
ب


In [19]:
# Unicode Encoding
s = 'ب'
print(s.encode("utf-8"))
print(list(s.encode("utf-8")))
print(b'\xd8\xa8'.decode("utf-8"))


b'\xd8\xa8'
[216, 168]
ب


In [34]:
# print(b"\xc3\x28".decode("utf-8"))
# UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 0: invalid continuation byte
# 0xC3 indicates the start of a two-byte UTF-8 character, but the following byte 0x28 is not a valid continuation byte 

### Byte-Pair Encoding (BPE) Tokenizer

In [47]:

text = """
u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.
Tom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."
Sam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."
They went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one could hear them. They were sad and scared, and they never got out of the pit.
<|endoftext|>
"""
special_tokens = ['<|endoftext|>']
split_pat = '|'.join(re.escape(st) for st in sorted(special_tokens, key=len, reverse=True))
parts = re.split(split_pat, text)
print(split_pat)
print(parts)

<\|endoftext\|>
['\nu don\'t have to be scared of the loud dog, I\'ll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.\n', '\nOnce upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.\nTom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."\nSam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."\nThey went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but n

In [51]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
regex.findall(PAT, "some text that i'll pre-tokenize")
# re.finditer(PAT, "some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

In [54]:
word_counts: dict = defaultdict(int)
for part in parts:
    for token in regex.findall(PAT, part):
        word = tuple(bytes([b]) for b in token.encode('utf-8'))
        word_counts[word] += 1
print(word_counts)

defaultdict(<class 'int'>, {(b'\n',): 8, (b'u',): 1, (b' ', b'd', b'o', b'n'): 1, (b"'", b't'): 1, (b' ', b'h', b'a', b'v', b'e'): 1, (b' ', b't', b'o'): 9, (b' ', b'b', b'e'): 1, (b' ', b's', b'c', b'a', b'r', b'e', b'd'): 3, (b' ', b'o', b'f'): 2, (b' ', b't', b'h', b'e'): 15, (b' ', b'l', b'o', b'u', b'd'): 1, (b' ', b'd', b'o', b'g'): 1, (b',',): 14, (b' ', b'I'): 1, (b"'", b'l', b'l'): 1, (b' ', b'p', b'r', b'o', b't', b'e', b'c', b't'): 1, (b' ', b'y', b'o', b'u'): 1, (b'"', b'.'): 1, (b' ', b'T', b'h', b'e'): 2, (b' ', b'm', b'o', b'l', b'e'): 3, (b' ', b'f', b'e', b'l', b't'): 1, (b' ', b's', b'o'): 1, (b' ', b's', b'a', b'f', b'e'): 2, (b' ', b'w', b'i', b't', b'h'): 1, (b' ', b'l', b'i', b't', b't', b'l', b'e'): 2, (b' ', b'g', b'i', b'r', b'l'): 1, (b'.',): 20, (b' ', b'S', b'h', b'e'): 1, (b' ', b'w', b'a', b's'): 6, (b' ', b'v', b'e', b'r', b'y'): 2, (b' ', b'k', b'i', b'n', b'd'): 1, (b' ', b'a', b'n', b'd'): 10, (b' ', b's', b'o', b'o', b'n'): 1, (b' ', b'c', b'a', b'm',

In [ ]:
vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
for i, st in enumerate(special_tokens):
    vocab[256 + i] = st.encode('utf-8')
print(vocab) # len(vocab) = 257

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

In [56]:
pair_counts: dict = defaultdict(int)
for word, count in word_counts.items():
    for i in range(len(word) - 1):
        pair_counts[(word[i], word[i + 1])] += count
print(pair_counts)

defaultdict(<class 'int'>, {(b' ', b'd'): 6, (b'd', b'o'): 2, (b'o', b'n'): 4, (b"'", b't'): 1, (b' ', b'h'): 13, (b'h', b'a'): 2, (b'a', b'v'): 1, (b'v', b'e'): 4, (b' ', b't'): 35, (b't', b'o'): 14, (b' ', b'b'): 16, (b'b', b'e'): 2, (b' ', b's'): 19, (b's', b'c'): 4, (b'c', b'a'): 6, (b'a', b'r'): 12, (b'r', b'e'): 9, (b'e', b'd'): 14, (b' ', b'o'): 5, (b'o', b'f'): 2, (b't', b'h'): 22, (b'h', b'e'): 37, (b' ', b'l'): 9, (b'l', b'o'): 6, (b'o', b'u'): 8, (b'u', b'd'): 1, (b'o', b'g'): 1, (b' ', b'I'): 2, (b"'", b'l'): 1, (b'l', b'l'): 10, (b' ', b'p'): 12, (b'p', b'r'): 1, (b'r', b'o'): 1, (b'o', b't'): 4, (b't', b'e'): 2, (b'e', b'c'): 1, (b'c', b't'): 1, (b' ', b'y'): 1, (b'y', b'o'): 1, (b'"', b'.'): 1, (b' ', b'T'): 15, (b'T', b'h'): 10, (b' ', b'm'): 6, (b'm', b'o'): 3, (b'o', b'l'): 3, (b'l', b'e'): 7, (b' ', b'f'): 11, (b'f', b'e'): 4, (b'e', b'l'): 4, (b'l', b't'): 1, (b's', b'o'): 2, (b's', b'a'): 6, (b'a', b'f'): 2, (b' ', b'w'): 14, (b'w', b'i'): 1, (b'i', b't'): 13, (b'l

In [ ]:
vocab_size = 300
num_merges = vocab_size - 256 - len(special_tokens)

In [ ]:
def train_bpe(input_path: str, vocab_size: int, special_tokens: list[str]):

    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Split on special tokens so they are never split by the pre-tokenizer
    if special_tokens:
        split_pat = '|'.join(re.escape(st) for st in sorted(special_tokens, key=len, reverse=True))
        parts = re.split(split_pat, text)
    else:
        parts = [text]

    # Count frequency of each pre-tokenized word (as a tuple of single-byte tokens)
    word_counts: dict = defaultdict(int)
    for part in parts:
        for token in regex.findall(PAT, part):
            word = tuple(bytes([b]) for b in token.encode('utf-8'))
            word_counts[word] += 1

    # Initialize vocab: indices 0-255 are single bytes, then special tokens
    vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
    for i, st in enumerate(special_tokens):
        vocab[256 + i] = st.encode('utf-8')

    # Build initial pair counts
    pair_counts: dict = defaultdict(int)
    for word, count in word_counts.items():
        for i in range(len(word) - 1):
            pair_counts[(word[i], word[i + 1])] += count

    num_merges = vocab_size - 256 - len(special_tokens)
    merges: list[tuple[bytes, bytes]] = []
    next_id = 256 + len(special_tokens)

    for _ in range(num_merges):
        if not pair_counts:
            break

        # Pick best pair: highest count, break ties lexicographically
        best_pair = max(pair_counts, key=lambda p: (pair_counts[p], p))

        new_token = best_pair[0] + best_pair[1]
        merges.append(best_pair)
        vocab[next_id] = new_token
        next_id += 1

        # Merge best_pair in every word, updating pair counts incrementally
        new_words: dict = {}
        for word, count in word_counts.items():
            if best_pair[0] not in word:
                new_words[word] = new_words.get(word, 0) + count
                continue

            new_word: list = []
            i = 0
            while i < len(word):
                if (i + 1 < len(word)
                        and word[i] == best_pair[0]
                        and word[i + 1] == best_pair[1]):
                    # Update left-neighbor pair
                    if new_word:
                        pair_counts[(new_word[-1], best_pair[0])] -= count
                        pair_counts[(new_word[-1], new_token)] += count
                    # Update right-neighbor pair (crosses the merge boundary)
                    if i + 2 < len(word):
                        pair_counts[(best_pair[1], word[i + 2])] -= count
                        pair_counts[(new_token, word[i + 2])] += count
                    new_word.append(new_token)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1

            t = tuple(new_word)
            new_words[t] = new_words.get(t, 0) + count

        del pair_counts[best_pair]
        word_counts = new_words

    return vocab, merges

In [ ]:
# _PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

# def _count_part(part):
#     counts = defaultdict(int)
#     for match in regex.finditer(_PAT, part):
#         word = tuple(bytes([b]) for b in match.group().encode('utf-8'))
#         counts[word] += 1
#     return dict(counts)

# def train_bpe(input_path: str, vocab_size: int, special_tokens: list[str]):

#     with open(input_path, 'r', encoding='utf-8') as f:
#         text = f.read()

#     # Split on special tokens so they are never split by the pre-tokenizer
#     if special_tokens:
#         split_pat = '|'.join(re.escape(st) for st in sorted(special_tokens, key=len, reverse=True))
#         parts = re.split(split_pat, text)
#     else:
#         parts = [text]

#     # Count frequency of each pre-tokenized word (as a tuple of single-byte tokens)
#     word_counts: dict = defaultdict(int)
#     with __import__('multiprocessing').Pool() as pool:
#         for partial_counts in pool.imap(_count_part, parts):
#             for word, count in partial_counts.items():
#                 word_counts[word] += count

#     # Initialize vocab: indices 0-255 are single bytes, then special tokens
#     vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
#     for i, st in enumerate(special_tokens):
#         vocab[256 + i] = st.encode('utf-8')

#     # Build initial pair counts
#     pair_counts: dict = defaultdict(int)
#     for word, count in word_counts.items():
#         for i in range(len(word) - 1):
#             pair_counts[(word[i], word[i + 1])] += count

#     num_merges = vocab_size - 256 - len(special_tokens)
#     merges: list[tuple[bytes, bytes]] = []
#     next_id = 256 + len(special_tokens)

#     for _ in range(num_merges):
#         if not pair_counts:
#             break

#         # Pick best pair: highest count, break ties lexicographically
#         best_pair = max(pair_counts, key=lambda p: (pair_counts[p], p))

#         new_token = best_pair[0] + best_pair[1]
#         merges.append(best_pair)
#         vocab[next_id] = new_token
#         next_id += 1

#         # Merge best_pair in every word, updating pair counts incrementally
#         new_words: dict = {}
#         for word, count in word_counts.items():
#             if best_pair[0] not in word:
#                 new_words[word] = new_words.get(word, 0) + count
#                 continue

#             new_word: list = []
#             i = 0
#             while i < len(word):
#                 if (i + 1 < len(word)
#                         and word[i] == best_pair[0]
#                         and word[i + 1] == best_pair[1]):
#                     # Update left-neighbor pair
#                     if new_word:
#                         pair_counts[(new_word[-1], best_pair[0])] -= count
#                         pair_counts[(new_word[-1], new_token)] += count
#                     # Update right-neighbor pair (crosses the merge boundary)
#                     if i + 2 < len(word):
#                         pair_counts[(best_pair[1], word[i + 2])] -= count
#                         pair_counts[(new_token, word[i + 2])] += count
#                     new_word.append(new_token)
#                     i += 2
#                 else:
#                     new_word.append(word[i])
#                     i += 1

#             t = tuple(new_word)
#             new_words[t] = new_words.get(t, 0) + count

#         del pair_counts[best_pair]
#         word_counts = new_words

#     return vocab, merges

In [60]:
import os
x = os.cpu_count()
print(x)

12


### Tokenizer

In [63]:
s = "\u0116"
s = "\u0120ex"
s = "\u111111"
print(s)

ᄑ11


### Linear

In [5]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super(Linear, self).__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features, device=device, dtype = dtype)) # nn.Parameter(...) # device=device # dtype = dtype
        std = math.sqrt(2/(in_features+out_features))
        a, b = -3*std, 3*std
        nn.init.trunc_normal_(self.weight, mean=0., std=std, a=a, b=b)
    def forward(self, x: torch.Tensor):
        return x@self.weight.T

In [367]:
## Test Linear class
x = torch.tensor([[1.,4.],[5.,6.]])
model = Linear(in_features=2, out_features=3)
y = model(x)
print(y)
print('-------------------------------------')

## Test load_state_dict
w = torch.tensor([[1., 2.],[3., 4.],[5., 6.]])
print('Before:', model.weight)
model.load_state_dict({"weight": w})
print('After: ', model.weight)

tensor([[ 1.3937, -3.1027,  1.1144],
        [-0.0972, -5.6129, -0.2389]], grad_fn=<MmBackward0>)
-------------------------------------
Before: Parameter containing:
tensor([[-0.6251,  0.5047],
        [-0.2740, -0.7072],
        [-0.5459,  0.4151]], requires_grad=True)
After:  Parameter containing:
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]], requires_grad=True)


### Embedding

In [6]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, device=None, dtype=None):
        super(Embedding, self).__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.weight = nn.Parameter(torch.empty(num_embeddings, embedding_dim, **factory_kwargs))
        nn.init.trunc_normal_(self.weight, mean=0, std=1, a=-3, b=3)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.weight[token_ids,:]

In [369]:
## Test Embedding class
embedding = Embedding(num_embeddings=10, embedding_dim=5)
token_ids = torch.tensor([4,5,6,7])
y = embedding(token_ids)
print(y)

tensor([[-0.5323,  1.1869,  0.7695, -0.3215, -0.8777],
        [-0.4229,  0.2562,  0.0680, -0.4745,  0.8756],
        [-1.1066,  0.6942,  0.2156, -0.5980, -0.1586],
        [ 0.9800,  2.1241,  1.6891, -0.0584,  0.5410]],
       grad_fn=<IndexBackward0>)


### RMSNorm

In [7]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device=None, dtype=None):
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.weight = nn.Parameter(torch.ones(d_model, **factory_kwargs ))
        self.eps = eps
    def forward(self, x):
        in_dtype = x.dtype
        x = x.to(torch.float32)
        rms = torch.rsqrt((x ** 2).mean(dim=-1, keepdim=True) + self.eps)
        return (x * rms * self.weight).to(in_dtype)

In [272]:
rmsnorm = RMSNorm(d_model = 2)
x = torch.tensor([[[1., 2.],[3.,4.]],[[5.,6.],[7.,8.]],[[9., 10.],[11.,12.]]])
y = rmsnorm(x)

In [274]:
t = torch.tensor([[[1., 2.],[3.,4.]],[[5.,6.],[7.,8.]],[[9., 10.],[11.,12.]]])
g = torch.tensor([0.5, 0.4, 0.3])
y = t*g.view(3,1,1) # [[[0.5000]],[[0.4000]],[[0.3000]]]
print(y)

tensor([[[0.5000, 1.0000],
         [1.5000, 2.0000]],

        [[2.0000, 2.4000],
         [2.8000, 3.2000]],

        [[2.7000, 3.0000],
         [3.3000, 3.6000]]])


### SwiGLU

In [8]:
# class SwiGLU(nn.Module):
#     def __init__(self, d_model: int, d_ff: int, device=None, dtype=None):
#         super().__init__()
#         # dff = int(((8./3. * d_model)//64)*64)
#         self.w1 = Linear(d_ff, d_model, device=device, dtype = dtype)
#         self.w2 = nn.Parameter(torch.randn(d_model, d_ff, device=device, dtype = dtype))
#         self.w3 = nn.Parameter(torch.randn(d_ff, d_model, device=device, dtype = dtype))
#     def forward(self, x):
#         return (((x@self.w1.T) * torch.sigmoid(x@self.w1.T))*(x@self.w3.T))@self.w2.T

class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int, device=None, dtype=None):
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.w1 = Linear(d_model, d_ff, **factory_kwargs)
        self.w3 = Linear(d_model, d_ff, **factory_kwargs)
        self.w2 = Linear(d_ff, d_model, **factory_kwargs)

    def forward(self, x):
        swish = self.w1(x) * torch.sigmoid(self.w1(x))
        gate = self.w3(x)
        return self.w2(swish * gate)
        #return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [ ]:
## Test SwiGLU class
model = SwiGLU(d_model = 64, d_ff = 128)
x = torch.randn(4,12,64)
y = model(x)
print(y.shape)

torch.Size([4, 12, 64])


### RotaryPositionalEmbedding

In [9]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device=None):
        super().__init__()
        inv_freq = 1./theta**(torch.arange(0,d_k,2,device=device)/d_k)
        ps = torch.arange(max_seq_len, device=device)
        angs = torch.outer(ps, inv_freq)
        self.d_k = d_k
        self.register_buffer('cos', torch.cos(angs))
        self.register_buffer('sin', torch.sin(angs))
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        token_positions = token_positions.to(x.device)
        sin = self.sin[token_positions] # [..., seq_len, d_k//2]
        cos = self.cos[token_positions] # [..., seq_len, d_k//2]
        x_even, x_odd = x[...,0::2], x[...,1::2]
        rope = torch.empty_like(x)
        rope[...,0::2], rope[...,1::2] = x_even * cos - x_odd * sin, x_odd * cos + x_even * sin
        return rope

In [278]:
x = torch.arange(24).view(2,3,4)
x_even, x_odd = x[...,0::2], x[...,1::2]
b = torch.empty_like(x)
b[...,0::2], b[...,1::2] = x_odd, -x_even
print(b)

tensor([[[  1,   0,   3,  -2],
         [  5,  -4,   7,  -6],
         [  9,  -8,  11, -10]],

        [[ 13, -12,  15, -14],
         [ 17, -16,  19, -18],
         [ 21, -20,  23, -22]]])


In [10]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device=None):
        super().__init__()
        inv_freq = 1./theta**(torch.arange(0,d_k,2)/d_k)
        ps = torch.arange(max_seq_len)
        angs = torch.outer(ps, inv_freq)
        self.d_k = d_k
        self.cos = torch.cos(angs)
        self.sin = torch.sin(angs)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        x1 = x[..., :self.d_k//2]   # [bs, seq_len, d_k//2]
        x2 = x[..., self.d_k//2:]   # [bs, seq_len, d_k//2]
        x_even = x1 * self.cos - x2 * self.sin
        x_odd  = x1 * self.sin + x2 * self.cos
        x_rope = torch.cat([x_even, x_odd], dim=-1)   # [bs, seq_len, d_k]
        return x_rope

In [280]:
bs, max_seq_len, d_k = 2, 5, 6 
theta = 1000
x = torch.randn(bs, max_seq_len, d_k)
inv_freq = 1./theta**(torch.arange(0,d_k,2)/d_k)
ps = torch.arange(max_seq_len)
angs = torch.outer(ps, inv_freq)
cos = torch.cos(angs)
sin = torch.sin(angs)
x1 = x[..., :d_k//2]   # [bs, seq_len, d_k//2]
x2 = x[..., d_k//2:]   # [bs, seq_len, d_k//2]

x_even = x1 * cos - x2 * sin
x_odd  = x1 * sin + x2 * cos

x_rope = torch.cat([x_even, x_odd], dim=-1)   # [bs, seq_len, d_k]

print(x_rope)

tensor([[[-1.3412,  2.0660, -0.1138, -0.3978, -0.0058, -1.7697],
         [-0.5590, -0.1132, -1.3856,  0.0896, -1.4457,  0.6053],
         [ 0.4840, -0.5092,  1.2577, -0.9000,  0.4186,  0.0156],
         [-0.3762, -0.5449,  1.1522,  0.7128, -0.3564, -0.5069],
         [-0.5689, -1.4189, -0.7205,  0.4982, -1.5114,  0.3414]],

        [[ 0.3092,  1.7914,  0.6914, -0.1572, -2.1741, -1.6705],
         [ 1.2864, -1.0515, -1.5611,  0.7659, -0.0099, -0.4869],
         [ 1.5355, -1.5497, -1.3027, -1.6584,  0.0399,  1.3769],
         [ 0.1587,  1.1709,  0.5845,  1.9496, -0.2858,  0.3103],
         [ 2.1345,  0.9014,  2.0674, -0.4132,  0.2953, -0.6623]]])


In [281]:
t = torch.arange(24).view(2,3,4)
m = torch.arange(3).view(3,1)
y = t*m
print(t)
print(m)
print(y)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
tensor([[0],
        [1],
        [2]])
tensor([[[ 0,  0,  0,  0],
         [ 4,  5,  6,  7],
         [16, 18, 20, 22]],

        [[ 0,  0,  0,  0],
         [16, 17, 18, 19],
         [40, 42, 44, 46]]])


In [282]:
t1 = t[..., :t.shape[-1]//2]
t2 = t[..., t.shape[-1]//2:]
t3 = torch.cat((-t2, t1),dim=-1)
print(t3)

tensor([[[ -2,  -3,   0,   1],
         [ -6,  -7,   4,   5],
         [-10, -11,   8,   9]],

        [[-14, -15,  12,  13],
         [-18, -19,  16,  17],
         [-22, -23,  20,  21]]])


In [283]:
a = torch.arange(4)
b = torch.arange(3)
c = torch.outer(b,a)
print(c)

tensor([[0, 0, 0, 0],
        [0, 1, 2, 3],
        [0, 2, 4, 6]])


In [284]:
theta, d_k = 1000, 6
inv_freq = 1./theta**(torch.arange(0,d_k,2)/d_k)
print(inv_freq)

tensor([1.0000, 0.1000, 0.0100])


In [285]:
torch.arange(0,d_k,2)/d_k

tensor([0.0000, 0.3333, 0.6667])

In [286]:
max_seq_len = 5
inv_freq = 1./theta**(torch.arange(0,d_k,2)/d_k)
ps = torch.arange(max_seq_len)
angs = torch.outer(ps, inv_freq)
print(angs)

tensor([[0.0000, 0.0000, 0.0000],
        [1.0000, 0.1000, 0.0100],
        [2.0000, 0.2000, 0.0200],
        [3.0000, 0.3000, 0.0300],
        [4.0000, 0.4000, 0.0400]])


In [287]:
inv_freq 

tensor([1.0000, 0.1000, 0.0100])

In [288]:
theta**(torch.arange(0,d_k,2))

tensor([            1,       1000000, 1000000000000])

### softmax

In [11]:
def softmax(x: torch.Tensor, dim: int):
    x_max = x.max(dim=dim, keepdim=True).values
    exp = torch.exp(x - x_max)
    return exp / exp.sum(dim=dim, keepdim=True)

x = torch.arange(24).view(2,3,4)
y = softmax(x, 0)
print(y)


tensor([[[6.1442e-06, 6.1442e-06, 6.1442e-06, 6.1442e-06],
         [6.1442e-06, 6.1442e-06, 6.1442e-06, 6.1442e-06],
         [6.1442e-06, 6.1442e-06, 6.1442e-06, 6.1442e-06]],

        [[9.9999e-01, 9.9999e-01, 9.9999e-01, 9.9999e-01],
         [9.9999e-01, 9.9999e-01, 9.9999e-01, 9.9999e-01],
         [9.9999e-01, 9.9999e-01, 9.9999e-01, 9.9999e-01]]])


In [44]:
print(x)
x_max = x.max(dim=-1, keepdim=True).values
print(x_max)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
tensor([[[ 3],
         [ 7],
         [11]],

        [[15],
         [19],
         [23]]])


### scaled_dot_product_attention

In [12]:
def scaled_dot_product_attention(Q, K, V, mask):
    d_k = Q.shape[-1]
    KT = K.transpose(-2,-1)
    S = Q@KT/math.sqrt(d_k)
    S = S.masked_fill(~mask, float('-inf'))
    A = softmax(S, dim = -1)
    return A@V

In [292]:
Q = torch.randn(2,2,2,64)
K = torch.randn(2, 2, 6, 64)
V = torch.randn(2, 2, 6, 64)
mask = torch.zeros(2, 2, 2, 6, dtype=torch.bool)
mask[...,0] = True
ans = scaled_dot_product_attention(Q, K, V, mask)

In [ ]:
seq, d_model = 3, 6
q = torch.arange(18).view(seq, d_model)
print(q)
print('-------------')
num_heads = 2
r = q.view(seq, -1, d_model//num_heads)
print(r)
print('-------------')
t = r.transpose(0, 1)       # [num_heads, seq_len, d_k]
print(t)

tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17]])
-------------
tensor([[[ 0,  1,  2],
         [ 3,  4,  5]],

        [[ 6,  7,  8],
         [ 9, 10, 11]],

        [[12, 13, 14],
         [15, 16, 17]]])
-------------
tensor([[[ 0,  1,  2],
         [ 6,  7,  8],
         [12, 13, 14]],

        [[ 3,  4,  5],
         [ 9, 10, 11],
         [15, 16, 17]]])


In [332]:
seq, d_model = 2, 12
q = torch.arange(24).view(seq, d_model)
print(q)
print('-------------')
num_heads = 3
d_head = d_model//num_heads
r = q.view(seq, num_heads, d_head)
print(r)
print('-------------')
t = r.transpose(0, 1)       # [num_heads, seq_len, d_head]
print(t)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]])
-------------
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
-------------
tensor([[[ 0,  1,  2,  3],
         [12, 13, 14, 15]],

        [[ 4,  5,  6,  7],
         [16, 17, 18, 19]],

        [[ 8,  9, 10, 11],
         [20, 21, 22, 23]]])


In [13]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, rope=None, device=None, dtype=None):
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        std = math.sqrt(1 / d_model)
        self.q_proj = Linear(d_model, d_model, **factory_kwargs)
        self.k_proj = Linear(d_model, d_model, **factory_kwargs)
        self.v_proj = Linear(d_model, d_model, **factory_kwargs)
        self.output_proj = Linear(d_model, d_model, **factory_kwargs)
        self.rope = rope

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None):
        seq_len = x.shape[1]
        Q = self._split_heads(self.q_proj(x))
        K = self._split_heads(self.k_proj(x))
        if self.rope is not None:
            assert token_positions is not None
            Q = self.rope(Q, token_positions)
            K = self.rope(K, token_positions)
        V = self._split_heads(self.v_proj(x))
        mask = torch.tril(torch.ones(1, 1, seq_len, seq_len, device=x.device)).bool()
        y = scaled_dot_product_attention(Q, K, V, mask)
        return self.output_proj(self._merge_heads(y))
    def _split_heads(self, t):
        bsz, seq_len = t.shape[:-1]
        return t.view(bsz, seq_len, self.num_heads, self.d_head).transpose(1, 2)
    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        bsz, _, seq_len, _ = x.shape
        return x.transpose(1, 2).contiguous().view(bsz, seq_len, self.d_model)


In [363]:
bs, t = 3, 4
d_model, num_heads = 6, 2
# x = torch.arange(72).view(bs, t, d_model).float()
x = torch.randn(bs,t,d_model)
model = MultiHeadSelfAttention(d_model, num_heads)
y = model(x)


In [ ]:
import torch
a = torch.ones(3, 3, dtype = bool)
x = torch.tril(a, diagonal=0)
print(x)

In [326]:
batch, seq, heads, head_dim = 2, 4, 2, 3
x = torch.arange(48).view(2,2,4,3) # batch * num_head * t * d_h 
print(x) 
y = x.transpose(1, 2).reshape(batch, seq, heads * head_dim)
print(y)

tensor([[[[ 0,  1,  2],
          [ 3,  4,  5],
          [ 6,  7,  8],
          [ 9, 10, 11]],

         [[12, 13, 14],
          [15, 16, 17],
          [18, 19, 20],
          [21, 22, 23]]],


        [[[24, 25, 26],
          [27, 28, 29],
          [30, 31, 32],
          [33, 34, 35]],

         [[36, 37, 38],
          [39, 40, 41],
          [42, 43, 44],
          [45, 46, 47]]]])
tensor([[[ 0,  1,  2, 12, 13, 14],
         [ 3,  4,  5, 15, 16, 17],
         [ 6,  7,  8, 18, 19, 20],
         [ 9, 10, 11, 21, 22, 23]],

        [[24, 25, 26, 36, 37, 38],
         [27, 28, 29, 39, 40, 41],
         [30, 31, 32, 42, 43, 44],
         [33, 34, 35, 45, 46, 47]]])


In [327]:
x = torch.tensor([0.7,0.7,1.41])
y = softmax(x, dim=0)
print(y)

tensor([0.2479, 0.2479, 0.5042])


In [336]:
bsz, seq_len = 3, 4
tp_1 = torch.arange(seq_len)
print(tp_1.shape)
tp_2 = torch.arange(seq_len).unsqueeze(0)
print(tp_2.shape)
tp_3 = torch.arange(seq_len).unsqueeze(0).expand(bsz, -1)
print(tp_3.shape)

torch.Size([4])
torch.Size([1, 4])
torch.Size([3, 4])


### TransformerBlock

In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_len, theta, eps: float = 1e-5, device=None):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.max_seq_len = max_seq_len
        self.theta = theta
        self.eps = eps
        self.device = device
        self.d_head = d_model // num_heads
        rope = RotaryPositionalEmbedding(self.theta, self.d_head, self.max_seq_len, self.device)
        self.attn = MultiHeadSelfAttention(self.d_model, self.num_heads, rope, self.device)
        self.ln1 = RMSNorm(d_model, eps, device)
        self.ln2 = RMSNorm(d_model, eps, device)
        self.ffn = SwiGLU(d_model, d_ff, device)
    def forward(self, x):
        token_positions = torch.arange(x.shape[1])
        y = x + self.attn(self.ln1(x), token_positions)
        y = y + self.ffn(self.ln2(y))

max_seq_len = 10
rope = RotaryPositionalEmbedding(theta, d_model//num_heads, max_seq_len)
d_ff = 10
model = TransformerBlock(d_model, num_heads, d_ff, max_seq_len, theta)
params = model.state_dict()
print(params.keys())

odict_keys(['attn.q_proj.weight', 'attn.k_proj.weight', 'attn.v_proj.weight', 'attn.output_proj.weight', 'ln1.weight', 'ln2.weight', 'ffn.w1.weight', 'ffn.w3.weight', 'ffn.w2.weight'])


### Transformer LM

In [ ]:

class TransformerLM(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, theta, context_length, vocab_size, num_layers, eps: float = 1e-5, device=None):
        super().__init__()
        self.token_embeddings = Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
        self.num_layers = num_layers
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, context_length, theta) 
            for _ in range(num_layers)])
        self.ln_final = RMSNorm(d_model, eps, device)
        self.lm_head = Linear(in_features=d_model, out_features=vocab_size)
    def forward(self, x):
        x = self.token_embeddings(x)
        for i in range(self.num_layers):
            x = self.layers[i](x)
        x = self.ln_final(x)
        x = self.lm_head (x)
        return x

d_model, num_heads, d_ff, theta, vocab_size, context_length, num_layers = 6, 2, 6, 1000, 25, 20, 3
model = TransformerLM(d_model, num_heads, d_ff, theta, context_length, vocab_size, num_layers)
params = model.state_dict()
print(params.keys())

odict_keys(['token_embeddings.weight', 'layers.0.attn.q_proj.weight', 'layers.0.attn.k_proj.weight', 'layers.0.attn.v_proj.weight', 'layers.0.attn.output_proj.weight', 'layers.0.ln1.weight', 'layers.0.ln2.weight', 'layers.0.ffn.w1.weight', 'layers.0.ffn.w3.weight', 'layers.0.ffn.w2.weight', 'layers.1.attn.q_proj.weight', 'layers.1.attn.k_proj.weight', 'layers.1.attn.v_proj.weight', 'layers.1.attn.output_proj.weight', 'layers.1.ln1.weight', 'layers.1.ln2.weight', 'layers.1.ffn.w1.weight', 'layers.1.ffn.w3.weight', 'layers.1.ffn.w2.weight', 'layers.2.attn.q_proj.weight', 'layers.2.attn.k_proj.weight', 'layers.2.attn.v_proj.weight', 'layers.2.attn.output_proj.weight', 'layers.2.ln1.weight', 'layers.2.ln2.weight', 'layers.2.ffn.w1.weight', 'layers.2.ffn.w3.weight', 'layers.2.ffn.w2.weight', 'ln_final.weight', 'lm_head.weight'])


### cross_entropy

In [ ]:
# instead of log(p) do this:
def cross_entropy(inputs, targets):
    logits = inputs - inputs.max(dim=-1, keepdim=True).values
    log_probs = logits - torch.log(torch.sum(torch.exp(logits), dim=-1, keepdim=True)) # you can use torch.logsumexp
    return -log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1).mean()

In [59]:
inputs = torch.randn(3,4)
targets = torch.tensor([1,3,2])

max_ = inputs.max(dim=-1, keepdim=True).values
inputs -= max_
zi = torch.gather(inputs, index = targets.unsqueeze(1), dim=-1)
zi.shape

torch.Size([3, 1])

### SGD

In [77]:
from collections.abc import Callable
from typing import Optional

class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()

        for group in self.param_groups:
            #print(group.keys())
            lr = group["lr"]  # Get the learning rate.

            for p in group["params"]:
                print(p)
                if p.grad is None:
                    continue

                state = self.state[p]  # Get state associated with p.
                t = state.get("t", 0)  # Get iteration number from the state, or initial value.
                grad = p.grad.data  # Get the gradient of loss with respect to p.

                p.data -= lr / math.sqrt(t + 1) * grad  # Update weight tensor in-place.
                state["t"] = t + 1  # Increment iteration number.
        print('___________')
        return loss

In [78]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=50)
for t in range(10):
    opt.zero_grad() # Reset the gradients for all learnable parameters. 
    loss = (weights**2).mean() # Compute a scalar loss value. 
    #print(loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients. 
    opt.step() # Run optimizer step.

Parameter containing:
tensor([[ -6.6236,  -3.9891,   0.0913,   5.1731,   5.7915,  -0.5158,  -5.4685,
          -0.8486,  -3.9248,   2.3934],
        [ -4.8215,  -3.5242,   2.8342,  -0.1065,  -0.9622,   5.3271,  -2.7590,
           5.9892,  -3.0206,   4.0076],
        [  6.2684,   1.5465,  -0.7783,   0.4772,  -1.6329,   2.2214,   0.4422,
           4.4971,  -3.2544,  -1.7092],
        [ -7.6648,   1.8106,   8.6110,  12.6199,   4.5494,  -3.8405,  -0.4487,
          -0.7981,   3.2514,  -3.7179],
        [  2.3177,  -3.0825,   1.3509,   0.5545,  -4.1914,   1.5643,  -0.0604,
          -8.5719,   3.4735,  -4.8482],
        [ -2.2076,  -1.1947,  -3.0391,   1.1633,   4.5992,  -1.4591,  -7.3866,
           0.5107, -11.4479,  -0.8246],
        [  2.6335,  -2.3770,  -1.6754,  -3.8293,   7.1473,  -7.0881,  -4.2629,
          -6.4592,   2.7447,  11.6578],
        [ -3.4802,  -4.2339,   2.0689,   7.2231,  -3.6210,   6.9091,   1.8893,
          -8.6799,  -2.4483,  -7.6087],
        [  3.9518,   1.227

### AdamW

In [ ]:
from collections.abc import Callable
from typing import Optional

class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr, beta1, eps, lnd):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr, }
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()

        for group in self.param_groups:
            lr = group["lr"]  # Get the learning rate.

            for p in group["params"]:
                
                if p.grad is None:
                    continue

                state = self.state[p]  # Get state associated with p.
                t = state.get("t", 0)  # Get iteration number from the state, or initial value.
                grad = p.grad.data  # Get the gradient of loss with respect to p.

                p.data -= lr / math.sqrt(t + 1) * grad  # Update weight tensor in-place.
                state["t"] = t + 1  # Increment iteration number.
        return loss

### Data Loader

In [ ]:
n = 10
x = np.array(range(n))
bs = 5
cntx = 4
smpls = np.random.randint(0, n-cntx, size= bs)
in_ = torch.empty((bs,cntx), dtype=torch.long)
out_ = torch.empty((bs,cntx), dtype=torch.long)
for cnt,e in enumerate(smpls):
    in_[cnt,:] = torch.tensor((x[e:e+cntx]))
    out_[cnt, :] = torch.tensor((x[e+1:e+1+cntx]))
ans = (in_, out_)


def data_loader(dataset, batch_size: int, context_length: int, device: str):
    n = len(dataset)
    smpls = np.random.randint(0, n - context_length, size=batch_size)
    indices = smpls[:, None] + np.arange(context_length + 1)[None, :]  # (batch_size, context_length+1)
    chunk = np.asarray(dataset)[indices]                               # (batch_size, context_length+1)
    in_ = torch.tensor(chunk[:, :-1], dtype=torch.long, device=device)
    out_ = torch.tensor(chunk[:, 1:], dtype=torch.long, device=device)
    return in_, out_


In [112]:
indices = smpls[:, None] + np.arange(cntx + 1)[None, :]
indices

array([[0, 1, 2, 3, 4],
       [1, 2, 3, 4, 5],
       [5, 6, 7, 8, 9],
       [2, 3, 4, 5, 6],
       [1, 2, 3, 4, 5]])

In [4]:
chr(0)
print(chr(0))

 


In [6]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [7]:
print("this is a test" + chr(0) + "string")

this is a test string


In [8]:
x = str("this is a test" + chr(0) + "string")

In [9]:
x

'this is a test\x00string'